<a href="https://colab.research.google.com/github/ShehabEddin64/Cue_Space/blob/main/MyTranscriptionTool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Étape 1 : Installation des dépendances
Cette cellule installe automatiquement les outils nécessaires pour extraire l'audio et générer les transcriptions (Whisper et FFmpeg).

In [ ]:
!pip -q install -U openai-whisper ffmpeg-python
!apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.3 MB/s eta 0:00:00


### Étape 2 : Configuration et exécution du traitement

#### Guide d'utilisation pour les non-développeurs :
1. **Optionnel** : Si vous le souhaitez, modifiez la valeur de `choix_modele` (par exemple, remplacez `"small"` par `"medium"` pour plus de précision mais plus de lenteur).
2. **Exécutez cette cellule** en cliquant sur l'icône de lecture (Play) à gauche.
3. **Sélectionnez vos fichiers** : Un bouton apparaîtra pour vous permettre de sélectionner une ou plusieurs vidéos depuis votre ordinateur.
4. **Téléchargement automatique** : Une fois le traitement de toutes les vidéos terminé, un fichier compressé `transcriptions.zip` contenant vos fichiers textes (.txt) et sous-titres (.srt) sera automatiquement téléchargé sur votre ordinateur.

In [ ]:
import whisper
import subprocess
import os
import shutil
from google.colab import files

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Sélectionnez la taille du modèle Whisper :
# - "tiny" : très rapide, précision faible
# - "base" ou "small" : rapide, bonne précision globale
# - "medium" ou "large" : très précis, plus lent (nécessite l'activation du GPU sous Colab)
choix_modele = "small"

# Langue et tâche :
# - "transcribe" : transcrit l'audio dans la langue d'origine de la vidéo.
# - "translate"  : traduit l'audio directement en anglais.
tache = "transcribe"

# Directory de sortie pour stocker les résultats temporairement
dossier_sortie = "transcriptions"
os.makedirs(dossier_sortie, exist_ok=True)

# ==============================================================================
# CHARGEMENT DU MODÈLE WHISPER
# ==============================================================================
print("Chargement du modèle de transcription en cours... (Veuillez patienter)")
model = whisper.load_model(choix_modele)
print("Modèle prêt pour la transcription.")

# ==============================================================================
# IMPORTATION DES FICHIERS VIDÉOS
# ==============================================================================
print(\n"Veuillez sélectionner les fichiers vidéos à traiter sur votre ordinateur :")
fichiers_charges = files.upload()
chemins_videos = list(fichiers_charges.keys())

print(f"\nNombre de vidéo(s) importée(s) : {len(chemins_videos)}")
for video in chemins_videos:
    print(" - ", video)

# ==============================================================================
# FONCTION FORMATAGE DES TIMESTAMPS (POUR LES SOUS-TITRES .SRT)
# ==============================================================================
def convertir_temps_srt(secondes):
    """Convertit un temps en secondes au format requis par les fichiers de sous-titres (HH:MM:SS,mmm)"""
    millisecondes = int(round(secondes * 1000))
    heures = millisecondes // 3600000
    millisecondes %= 3600000
    minutes = millisecondes // 60000
    millisecondes %= 60000
    secs = millisecondes // 1000
    millisecondes %= 1000
    return f"{heures:02d}:{minutes:02d}:{secs:02d},{millisecondes:03d}"

# ==============================================================================
# TRAITEMENT INDIVIDUEL DE CHAQUE VIDÉO
# ==============================================================================
videos_reussies = []
videos_echouees = []

for index, chemin_video in enumerate(chemins_videos, start=1):
    print("\n" + "-" * 50)
    print(f"Traitement de la vidéo {index}/{len(chemins_videos)}")
    print(f"Fichier : {chemin_video}")
    print("-" * 50)

    chemin_audio_temp = None
    try:
        nom_sans_extension = os.path.splitext(os.path.basename(chemin_video))[0]
        chemin_audio_temp = f"{nom_sans_extension}_audio.wav"

        # 1. Extraction de la piste audio de la vidéo avec l'utilitaire FFmpeg
        print("Extraction de la bande sonore...")
        subprocess.run([
            "ffmpeg", "-y", "-i", chemin_video,
            "-vn", "-ac", "1", "-ar", "16000", chemin_audio_temp
        ], check=True, capture_output=True)

        # 2. Lancement de l'analyse et transcription par l'intelligence artificielle
        print("Analyse et transcription de l'audio par Whisper...")
        resultat = model.transcribe(chemin_audio_temp, task=tache)

        # 3. Création des fichiers de sortie (Fichier texte brut et Fichier de sous-titres)
        fichier_txt = os.path.join(dossier_sortie, f"{nom_sans_extension} - Transcription.txt")
        fichier_srt = os.path.join(dossier_sortie, f"{nom_sans_extension} - Transcription.srt")

        # Enregistrement de la transcription textuelle simple
        with open(fichier_txt, "w", encoding="utf-8") as f:
            f.write(resultat["text"].strip() + "\n")

        # Enregistrement au format SRT avec repères temporels (sous-titres)
        with open(fichier_srt, "w", encoding="utf-8") as f:
            for i, segment in enumerate(resultat["segments"], start=1):
                f.write(f"{i}\n")
                f.write(f"{convertir_temps_srt(segment['start'])} --> {convertir_temps_srt(segment['end'])}\n")
                f.write(segment["text"].strip() + "\n\n")

        # Suppression du fichier audio de travail temporaire pour libérer de la mémoire
        if os.path.exists(chemin_audio_temp):
            os.remove(chemin_audio_temp)

        videos_reussies.append(chemin_video)
        print(f"Transcription terminée avec succès pour : {nom_sans_extension}")

    except Exception as erreur:
        videos_echouees.append(chemin_video)
        print(f"Erreur lors du traitement du fichier {chemin_video} : {erreur}")
        if chemin_audio_temp and os.path.exists(chemin_audio_temp):
            os.remove(chemin_audio_temp)

# ==============================================================================
# FINALISATION ET TÉLÉCHARGEMENT DE L'ARCHIVE ZIP
# ==============================================================================
print("\n" + "=" * 50)
print("TRAITEMENT DES VIDÉOS TERMINÉ")
print("=" * 50)

if videos_reussies:
    # Regroupement des transcriptions générées dans un dossier ZIP pour faciliter le téléchargement
    nom_archive = "transcriptions"
    shutil.make_archive(nom_archive, "zip", dossier_sortie)
    print(f"\nCréation du dossier compressé contenant vos résultats : {nom_archive}.zip")

    # Commande de téléchargement automatique sur le poste de l'utilisateur
    files.download(nom_archive + ".zip")

if videos_echouees:
    print("\nCertains fichiers n'ont pas pu être traités :")
    for v in videos_echouees:
        print(" - ", v)

Chargement du modèle Whisper : small


100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 89.5MiB/s]


✅ Modèle chargé et prêt.

Sélectionnez vos vidéos à traiter :


Saving Propaganda (12).mp4 to Propaganda (12).mp4
Saving Propaganda (13).mp4 to Propaganda (13).mp4
Saving Propaganda (14).mp4 to Propaganda (14).mp4
Saving Propaganda (15).mp4 to Propaganda (15).mp4
Saving Propaganda (1).mp4 to Propaganda (1).mp4
Saving Propaganda (2).mp4 to Propaganda (2).mp4
Saving Propaganda (3).mp4 to Propaganda (3).mp4
Saving Propaganda (4).mp4 to Propaganda (4).mp4
Saving Propaganda (5).mp4 to Propaganda (5).mp4
Saving Propaganda (6).mp4 to Propaganda (6).mp4
Saving Propaganda (7).mp4 to Propaganda (7).mp4
Saving Propaganda (8).mp4 to Propaganda (8).mp4
Saving Propaganda (9).mp4 to Propaganda (9).mp4
Saving Propaganda (10).mp4 to Propaganda (10).mp4
Saving Propaganda (11).mp4 to Propaganda (11).mp4
Saving Propaganda.mp4 to Propaganda.mp4

✅ 16 vidéo(s) chargée(s) :
 - Propaganda (12).mp4
 - Propaganda (13).mp4
 - Propaganda (14).mp4
 - Propaganda (15).mp4
 - Propaganda (1).mp4
 - Propaganda (2).mp4
 - Propaganda (3).mp4
 - Propaganda (4).mp4
 - Propaganda (5).mp

/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


...

✅ Succès : Propaganda (11)

TRAITEMENT VIDÉO 16/16
Nom du fichier : Propaganda.mp4
🔊 Extraction de la piste audio (WAV 16kHz mono)...
🧠 Analyse et transcription par l'IA...


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Succès : Propaganda

TERMINÉ : TOUS LES FICHIERS SONT TRAITÉS

📦 Archive créée : transcriptions.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>